In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
#%matplotlib widget

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
HE_I = 6680 #Reference: Johanna brain
HE_II = 4686 #Reference: Johanna brain

In [ ]:
#read in the data and deal with radial velocities
with open('hot_stars_subsample_2026-06-27_data.pkl','rb') as f:
     fluxes, loglam, ivars, continuua, spec_files, rvs = pickle.load(f)
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape, spec_files.shape)

In [ ]:
#load stellar parameters, check pkl and parquet shapes match

df = pd.read_parquet("hot_stars_subsample_2026-06-27_table.parquet")
print("num of stars in parquet:", df.shape[0])

In [ ]:
#log lambda values have been corrupted
print(10**loglam)
print(loglam[1:] - loglam[:-1])
print(np.median(loglam[1:] - loglam[:-1]))
delta_log_lambda_pixel = np.median(loglam[1:] - loglam[:-1])
wave = 10**loglam

In [ ]:
#get ready to do pixel shifts
delta_log_lambdas = (rvs / c) / LN10
delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

print(delta_log_lambdas, delta_pixels)

In [ ]:
#shift spectra to the rest frame
## johanna is not going to like this

rest_fluxes = np.zeros_like(fluxes) + 1.
rest_ivars = np.zeros_like(ivars)
rest_continuua = np.zeros_like(continuua) + np.nan
for i, dp in enumerate(delta_pixels):
    if dp < 0:
        rest_fluxes[i, -dp:] = fluxes[i, :dp]
        rest_ivars[i, -dp:] = ivars[i, :dp]
        rest_continuua[i, -dp:] = continuua[i, :dp]
        
    elif dp > 0:
        rest_fluxes[i, :-dp] = fluxes[i, dp:]
        rest_ivars[i, :-dp] = ivars[i, dp:]
        rest_continuua[i, :-dp] = continuua[i, dp:]
        
    else:
        rest_fluxes[i, :] = fluxes[i, :]
        rest_ivars[i, :] = ivars[i, :]
        rest_continuua[i, :] = continuua[i, :]


In [ ]:
#fix nans and infinities
bad = np.logical_not(np.isfinite(rest_fluxes))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_not(np.isfinite(rest_ivars))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_or((rest_fluxes > 2.0), (rest_fluxes < 0))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = rest_ivars > MAX_IVAR
# rest_fluxes[bad] = 1
# rest_ivars[bad] = 0
rest_ivars[bad] = MAX_IVAR

bad = ivars < MIN_IVAR
rest_fluxes[bad] = 1.0
rest_ivars[bad] = MIN_IVAR

In [ ]:
#we are using the rest frame data as our data 
data = rest_fluxes
weights = rest_ivars

In [ ]:
#split data to A and B, only train on A
N, M = fluxes.shape
foo = np.arange(N)
rng = np.random.default_rng(17)
rng.shuffle(foo)
Aindx = foo[:N // 2]
Bindx = foo[N // 2:]

#hyperparameters set here
K = 12
modelA = Robusta(rank=K, robust=True, robust_scale=1, robust_nu=1)
start = time.perf_counter()
modelA.fit(data[Aindx], weights[Aindx], max_iter=10000)
end = time.perf_counter()
print("total time:", (end - start)/60, "minutes")

In [ ]:
# synthesize, but masking Balmer lines.
log_H_ALPHA, log_H_BETA, log_HE_I, log_HE_II = np.log10(H_ALPHA), np.log10(H_BETA), np.log10(HE_I), np.log10(HE_II)

H_delta_lnlam_line = 1000 / c # km/s, this is the v/c ratio notice must be natural log for ln(x + 1) ≈ x to hold
H_delta_loglam_line = H_delta_lnlam_line / LN10 #convert to log10 units must match!!
region_alpha = np.abs(loglam - log_H_ALPHA)
region_beta = np.abs(loglam - log_H_BETA)

#do the same but half width of 500km/s for helium lines
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
region_he_i = np.abs(loglam - log_HE_I)
region_he_ii = np.abs(loglam - log_HE_II)


censor_mask = np.ones_like(loglam)
censor_mask[(region_alpha < H_delta_loglam_line) | (region_beta < H_delta_loglam_line) 
          | (region_he_i < HE_delta_loglam_line) | (region_he_ii < HE_delta_loglam_line)] = 0 #?

state, _ = modelA.infer(data[Bindx], weights[Bindx] * censor_mask[None, :])
synth_B = modelA.synthesize(state)

In [ ]:
# check that mask is not wrong
plt.plot(10**loglam, censor_mask, 'k')
plt.axvline(H_ALPHA, alpha = 0.5, color = 'violet')
plt.axvline(H_BETA, alpha = 0.5, color = 'violet')
plt.axvline(HE_I, alpha = 0.5)
plt.axvline(HE_II, alpha = 0.5)
plt.xlim(4200,7000)
plt.show()

## Eigenspectra Plotting

In [ ]:
eigsA = modelA.basis_vectors().T

f = plt.figure(figsize=(15, 15))
plt.title("Eigenspectra")
offset = 0.2
for i in range(K):
    f.gca().plot(wave, eigsA[i] + i * offset)

#plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="blue")
plt.xlabel("Wavelength (angstrom)")
plt.show()

## Synthesized Spectra Examples

In [ ]:
# look at a few spectra and syntheses
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[i]] + i * offset, color="k")
    f.gca().plot(wave, synth_B[i] + i * offset, color="r")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_I, lw=1, alpha=0.5, color="b")
plt.axvline(HE_II, lw=1, alpha=0.5, color="b")
plt.xlim(4200, 7000)
plt.xlabel("Wavelength (angstrom)")
plt.show()

## Residual Examples

In [ ]:
# look at residuals at H-alpha
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[i]] + i * offset - synth_B[i], color="k")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength (angstrom)")
#plt.xlim(H_ALPHA - 200., H_ALPHA + 200.)
plt.title("Residual examples")
plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_I, lw=1, alpha=0.5, color="b")
plt.axvline(HE_II, lw=1, alpha=0.5, color="b")
plt.xlim(4200, 7000)
plt.show()

### measure equivalent widths

In [ ]:
#HOGG PSUEDO CODE
# note: This depends on whether or not we are using continuum-normalized spectra.
#       Here we assume that we are continuum normalized.
# algorithm:
# - get residuals
# - choose pixels
# - sum(residual * delta_log_lambda_pixel * LN10 * wavelength) in the pixels


#Notes/explanation for Nana:
#lam is true wavelength values (not evenly spaced), delta_log_lambda_pixel (evenly spaced)
#Integration weight converts everything to angrstrom (delta_log_lambda_pixel * LN10 * lam [angstrom])
# " * LN10" here converts log10 --> ln, 
# " * lam" in order to get delta_lambda recall: Δλ=λ×Δlnλ
#np.abs(logline - loglam) < delta_loglam_line) restricts the width of the summation

In [ ]:
#Recall how we defined some stuff in the censor masking section
# H_delta_lnlam_line = 1000 / c #1000km/s half width for H_ALPHA, H_BETA
# H_delta_loglam_line = H_delta_lnlam_line / LN10 
# HE_delta_lnlam_line = 500 / c #500km/s half width for HE_I, HE_II
# HE_delta_loglam_line = HE_delta_lnlam_line / LN10


lam = 10 ** loglam
N, M = data[Bindx].shape
lines = [H_ALPHA, H_BETA, HE_I, HE_II]  
#make sure half_widths and lines match!!!!!!
half_wids = [H_delta_loglam_line, H_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 
ews = np.zeros((N, len(lines))) + np.nan

for j, (line, wid) in enumerate(zip(lines, half_wids)):
    logline = np.log10(line)
    integration_weight = delta_log_lambda_pixel * LN10 * lam * (np.abs(logline - loglam) < wid)
    ews[:, j] = np.sum((data[Bindx] - synth_B) * integration_weight[None, :], axis=1)

## Scatter Plots of EWS

In [ ]:
plt.plot(ews[:, 0], ews[:, 1], "k.")
plt.xlabel("H alpha EW")
plt.ylabel("H beta EW")
plt.show()

plt.plot(ews[:, 0], ews[:, 2], "k.")
plt.xlabel("H alpha EW")
plt.ylabel("He I EW")
plt.show()

plt.plot(ews[:, 0], ews[:, 3], "k.")
plt.xlabel("H alpha EW")
plt.ylabel("He II EW")
plt.show()

## Sorted by highest EW, Residuals

In [ ]:
#sort the synthesized 
foo = np.argsort(ews[:, 0])[::-1]
f = plt.figure(figsize=(10, 8))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[foo[i]]] + i * offset - synth_B[foo[i]], color="k")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength")
plt.xlim(H_ALPHA - 200., H_ALPHA + 200.)
halpha_lo = 10**(log_H_ALPHA - H_delta_loglam_line)
halpha_hi = 10**(log_H_ALPHA + H_delta_loglam_line)
plt.axvspan(halpha_lo, halpha_hi, color='royalblue', alpha=0.3)
hbeta_lo = 10**(log_H_BETA - H_delta_loglam_line)
hbeta_hi = 10**(log_H_BETA + H_delta_loglam_line)
plt.axvspan(hbeta_lo, hbeta_hi, color='royalblue', alpha=0.3)
plt.title("12 Highest EWs, residual near Halpha")
plt.show()

## Sorted by highest EW, synthesized spectra

In [ ]:
# look at a few spectra and syntheses
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(12):
    f.gca().plot(wave, data[Bindx[foo[i]]] + i * offset, color="k")
    f.gca().plot(wave, synth_B[foo[i]] + i * offset, color="r")

plt.axvline(H_ALPHA, lw=1, alpha=0.5, color="b")
plt.axvline(H_BETA, lw=1, alpha=0.5, color="b")
plt.axvline(HE_I, lw=1, alpha=0.5, color="b")
plt.xlabel("Wavelength")
plt.xlim(H_ALPHA - 200, H_ALPHA + 200) 
plt.show()